# NLST DICOM Preprocessing

Converts NLST DICOM series into JPG images for external validation, selecting a central subset of slices per patient rather than the full scan.

## Input structure
DICOM files organized as `NLST_root/{CC,NOC}/{patient_id}/...`, where `CC` denotes confirmed cancer cases and `NOC` denotes non-cancer cases.

## Processing steps
1. Flatten patient folders: move all DICOM files into a single directory per patient, renamed as `{patient_id}_{original_name}.dcm`.
2. Select the central 54% of slices per patient (excluding the first 18% and last 28%), to retain the anatomical region most likely to contain the region of interest while discarding boundary slices.
3. Convert selected DICOM files to JPG using HU windowing (window level -600, window width 1500).
4. Consolidate output into `NLST_CENTRAL_JPG/{CC,NOC}/{patient_id}/{patient_id}_{slice_name}.jpg`.

## Settings
- `wl` and `ww` control the HU windowing range; defaults match the values used for the reported dataset.
- Slice retention percentages (18% / 54% / 28%) are fixed at the top of the slice-selection function and can be adjusted for a different central-slice ratio.

In [ ]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'max_split_size_mb:64'

import torch

# Comprobar si hay una GPU disponible, cuál es y seleccionar "device 0"
if torch.cuda.is_available():
    device = torch.device('cuda:1')
    print(f"GPU disponible: {torch.cuda.get_device_name(1)}")
else:
    device = torch.device('cpu')
    print("GPU no disponible, usando CPU.") 

# Configurar fastai para usar el dispositivo específico
#hide
from fastai.vision.all import *
defaults.device = device
device

In [ ]:
from comet_ml import Experiment
from fastai.callback.comet import CometCallback


# Importar otras bibliotecas después de Comet.ml
from fastai.vision.all import *
from fastai.basics import *
from fastai.callback.all import *
from fastai.medical.imaging import *
from fastai.data.transforms import IndexSplitter 
from fastai.data.core import DataLoaders
from fastai.vision.data import PILImage
from PIL import Image   
#import pydicom
from fastai.learner import Learner
from fastai.losses import BCEWithLogitsLossFlat
# from fastai.callback.progress import ProgressCallback
# from fastai.callback.comet import CometCallback

#import torch
#import torch.nn as nn
import torch.nn.functional as F

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score, recall_score, precision_score, f1_score
import requests
import io
import pandas as pd
import numpy as np

from pathlib import Path
import shutil
import pydicom

import cv2



## Input structure

In [ ]:
# Function to organize the folders

def flatten_patient_folders(root_dir):
    """
    root_dir: Path a NLCT_cc o NLCT_noc
    """

    root_dir = Path(root_dir)

    for patient_dir in root_dir.iterdir():
        if not patient_dir.is_dir():
            continue

        patient_id = patient_dir.name

        #Search for all DICOM images within (even if nested)
        dicom_files = list(patient_dir.rglob("*.dcm"))

        if len(dicom_files) == 0:
            print(f"[WARNING] No DICOMs found for patient {patient_id}")
            continue

        for dicom_path in dicom_files:
            # New name: PatientID_XXXX.dcm
            original_name = dicom_path.name
            new_name = f"{patient_id}_{original_name}"

            target_path = patient_dir / new_name

            # Avoid accidental overwriting
            if target_path.exists():
                print(f"[SKIP] {target_path} already exists")
                continue

            shutil.move(str(dicom_path), str(target_path))

        # Cleanup: delete empty folders (StudyID / SeriesID)
        for subdir in sorted(patient_dir.rglob("*"), reverse=True):
            if subdir.is_dir() and not any(subdir.iterdir()):
                subdir.rmdir()

        print(f"[OK] Patient {patient_id}: {len(dicom_files)} images processed")



##### NOTE: This function cannot be run again, as it changes the file names.

In [ ]:

#Primero uno
#flatten_patient_folders("/home2/rosmeri/Fastai_Kaggle_Desarrollando/2daBBDD_Lung-Cancer_/Phase0_crear_clasificador_plus_predicción/Experimento_Modelo/Testing_la-otra-bbdd/nlst/NLST_cc/")

#Luego el otro
#flatten_patient_folders("/home2/rosmeri/Fastai_Kaggle_Desarrollando/2daBBDD_Lung-Cancer_/Phase0_crear_clasificador_plus_predicción/Experimento_Modelo/Testing_la-otra-bbdd/nlst/NLST_noc")


##### NOTE: Do not delete this; it may help make corrections if the function above is repeated.

In [ ]:
'''
from pathlib import Path
import re

def clean_duplicate_ids(root_dir):
    root_dir = Path(root_dir)

    for patient_dir in root_dir.iterdir():
        if not patient_dir.is_dir():
            continue

        patient_id = patient_dir.name

        for dicom_file in patient_dir.glob("*.dcm"):
            original_name = dicom_file.name
            stem = dicom_file.stem  # sin extensión

            # Detectar múltiples repeticiones del ID separadas por "_"
            parts = stem.split("_")

            # Contar cuántas veces aparece el patient_id al principio
            count = 0
            for part in parts:
                if part == patient_id:
                    count += 1
                else:
                    break

            if count > 1:
                # Eliminar repeticiones y reconstruir nombre limpio
                new_stem = "_".join([patient_id] + parts[count:])
                new_name = new_stem + ".dcm"
                new_path = dicom_file.parent / new_name

                # Renombrar
                dicom_file.rename(new_path)
                print(f"[RENOMBRADO] {original_name} → {new_name}")
            else:
                print(f"[OK] {original_name} ya está bien")

    print("\n Todos los nombres corregidos.")

# Usa la función con tu carpeta NOX
clean_duplicate_ids("/home2/rosmeri/Fastai_Kaggle_Desarrollando/2daBBDD_Lung-Cancer_/Phase0_crear_clasificador_plus_predicción/Experimento_Modelo/Testing_la-otra-bbdd/nlst/NLST_noc")
'''

In [1]:
##########################################################################################
# 1 Extract the central images (3) for each patient and group them into a specific file
########################################################################################
#
# 1.	Count the number of DICOM images
# 2.	Sort them correctly
# 3.	Select the 3 central slices
# 4.	Save them in a specific folder (e.g., CENTRAL_SLICES/)
# 5.	Retain the patient ID in the filename

# If n is odd (e.g., 101 images)
# 	mid = 50
#   Select: 49, 50, and 51

# If n is even (e.g., 100 images)
#  mid = 50
#  Select: 49, 50, and 51

In [ ]:
import pandas as pd
nlst_nueva = pd.read_csv("/home2.../nlst_nuevo_por_Rose.csv")
nlst_nueva

In [ ]:
import shutil
from pathlib import Path
import pandas as pd

# Base de datos resumen
df_info = nlst_nueva # here DataFrame here with columns: pid, has_cancer, anomalia_slice_num

# Base directories
BASE_DIRS = {
    "CC": Path("/home.../nlst/NLST_cc/"),
    "NOC": Path("/home.../nlst/NLST_noc/")
}
OUTPUT_DIR = Path("/home.../nlst/NLST_CENTRAL_SLICES")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def get_image_index(filename):
    """Extracts the slice number from a name like 100012_1-039.dcm."""
    try:
        return int(filename.stem.split("-")[-1])
    except:
        return -1

for label, base_dir in BASE_DIRS.items():
    print(f"\nProcessing {label}...")

    for patient_dir in base_dir.iterdir():
        if not patient_dir.is_dir():
            continue

        #patient_id = int(patient_dir.name)
        try:
            patient_id = int(patient_dir.name)
        except ValueError:
            print(f"   [SKIP] Ignored folder: {patient_dir.name}")
            continue

        out_patient_dir = OUTPUT_DIR / label / str(patient_id)
        out_patient_dir.mkdir(parents=True, exist_ok=True)

        dicom_files = sorted(patient_dir.glob("*.dcm"), key=get_image_index)

        if label == "CC":
            row = df_info.query(f"pid == {patient_id} and has_cancer == 1")
            if row.empty or pd.isna(row.iloc[0]["anomalia_slice_num"]):
                print(f"   {patient_id}: sin slice relevante o sin cáncer, omitido")
                continue

            target_slice = int(row.iloc[0]["anomalia_slice_num"])

           # Find file names with index = target_slice ± 1
            slices_to_copy = []
            for offset in [-1, 0, 1]:
                desired = target_slice + offset
                found = [f for f in dicom_files if get_image_index(f) == desired]
                if found:
                    slices_to_copy.append(found[0])

            for dicom_path in slices_to_copy:
                shutil.copy(dicom_path, out_patient_dir / dicom_path.name)

            print(f"   {patient_id}: copiado slice {target_slice} ±1")

        else:  # NOC
            if len(dicom_files) < 3:
                print(f"   {patient_id}: <3 images, omitted")
                continue
            mid = len(dicom_files) // 2
            central = dicom_files[mid - 1: mid + 2]
            for dcm in central:
                shutil.copy(dcm, out_patient_dir / dcm.name)
            print(f"   {patient_id}: 3 central units copied")


In [ ]:
# Read DICOM + HU
import pydicom

def load_dicom_hu(dicom_path):
    ds = pydicom.dcmread(dicom_path)
    img = ds.pixel_array.astype(np.float32)

    slope = float(ds.get("RescaleSlope", 1.0))
    intercept = float(ds.get("RescaleIntercept", -1024.0))

    img = img * slope + intercept
    return img


# Aplicar las ventanas # Aplicar las ventanas 
def apply_window(img, wl, ww):
    lower = wl - ww / 2
    upper = wl + ww / 2

    img = np.clip(img, lower, upper)
    return img

# Normalize to 8-bit

def normalize_to_uint8(img):
    img = img - img.min()
    img = img / img.max()
    img = (img * 255).astype(np.uint8)
    return img


# From Grayscale to RGB
def gray_to_rgb(img):
    return np.stack([img, img, img], axis=-1)

# DICOM - JPG

def dicom_to_jpg_rgb(dicom_path, output_path, wl, ww):  #wl=-600, ww=1500
    img = load_dicom_hu(dicom_path)
    img = apply_window(img, wl, ww)
    img = normalize_to_uint8(img)
    rgb = gray_to_rgb(img)

    #output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    
    cv2.imwrite(str(output_path), rgb)

#
# To generate using a single window
#

# Convert all patients at once (but now I want each one saved in its own folder again)
def convert_dataset(input_root, output_root, wl, ww):  #wl=-600, ww=1500
    input_root = Path(input_root)
    output_root = Path(output_root)

    for patient_dir in input_root.iterdir():
        if not patient_dir.is_dir():
            continue

        patient_id = patient_dir.name
        out_patient_dir = output_root / patient_id

        for dicom_file in patient_dir.glob("*.dcm"):
            out_name = dicom_file.stem + ".jpg"
            out_path = out_patient_dir / out_name

            dicom_to_jpg_rgb(
                dicom_file,
                out_path,
                wl=wl,
                ww=ww
            )

  

In [ ]:
# Para CC

convert_dataset(
    input_root="/home2.../nlst/NLST_CENTRAL_SLICES/CC",
    output_root="/home2.../nlst/NLST_CENTRAL_JPG/CC",
    wl=-400,
    ww=1200
    #wl=400,
    #ww=600
) 


# Para NOC
convert_dataset(
    input_root="/home/.../nlst/NLST_CENTRAL_SLICES/NOC",
    output_root="/home2.../nlst/NLST_CENTRAL_JPG/NOC",
    wl=600,
    ww=350 #600
) 


In [ ]:
# check image size

patient_1 = "/home.../nlst/NLST_CENTRAL_JPG/CC/100012"

patient_2= "/home.../nlst/NLST_CENTRAL_JPG/NOC/100020"


images_1 = sorted([f for f in os.listdir(patient_1) if f.endswith(".jpg")])

for img_name in images_1:
    img = Image.open(os.path.join(patient_1, img_name))
    print(img_name, img.size, img.mode)


images_2 = sorted([f for f in os.listdir(patient_2) if f.endswith(".jpg")])

for img_name in images_2:
    img = Image.open(os.path.join(patient_2, img_name))
    print(img_name, img.size, img.mode)


In [ ]:
images_1[1]

In [ ]:
images_2[1]

In [ ]:
img_1 = Image.open(os.path.join(patient_1, images_1[2]))
img_1

In [ ]:
img_2 = Image.open(os.path.join(patient_2, images_2[1]))
img_2